# Lesson 21: Graph Engineering — Nodes, Edges, Shared State & Routing

In this notebook, you will:
1. Build a 4-node Research → Implement → Verify → Merge execution graph.
2. Attach step-by-step checkpoints and fresh-context Verify isolation.

In [ ]:
!pip install -q -U google-genai langgraph

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class DevState(TypedDict):
    code: str
    passed: bool
    attempts: int

def impl_node(state: DevState) -> dict:
    nxt = state["attempts"] + 1
    return {"code": f"v{nxt}", "attempts": nxt}

def verify_node(state: DevState) -> dict:
    return {"passed": state["attempts"] >= 2}

g = StateGraph(DevState)
g.add_node("implement", impl_node)
g.add_node("verify", verify_node)
g.set_entry_point("implement")
g.add_edge("implement", "verify")
g.add_conditional_edges("verify", lambda s: END if s["passed"] else "implement")
app = g.compile()
out = app.invoke({"code": "", "passed": False, "attempts": 0})
print("LangGraph final state:", out)
assert out["passed"] and out["attempts"] == 2